In [1]:
# importing relevant libraries to complete all tasks
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from pathlib import Path
import os
import json
from fairlearn.metrics import (
    MetricFrame, selection_rate, true_positive_rate, false_positive_rate,
    count, demographic_parity_difference, equalized_odds_difference,
)

In [2]:
# Get the current notebook's directory
notebook_dir = Path().resolve() 

In [3]:
# csv file name declaration
data_file_1 = 'financial_fraud_detection_dataset.csv'
data_file_2 = 'FraudShield_Banking_Data (1).csv'

In [4]:
# Build the relative path to the CSV file
csv_file_path_1 = os.path.join(notebook_dir, data_file_1)
csv_file_path_2 = os.path.join(notebook_dir, data_file_2)

In [5]:
finanical_data_raw = pd.read_csv(csv_file_path_1, sep=',', decimal='.', header=0)

In [6]:
banking_data_raw = pd.read_csv(csv_file_path_2, sep=',', decimal='.', header=0)

In [7]:
# Before the data preparation, whole set of data will be cloned for further processing
finanical_data_process = finanical_data_raw.copy()

In [8]:
# Before the data preparation, whole set of data will be cloned for further processing
banking_data_process = banking_data_raw.copy()

In [9]:
def _add_time_features(df, dt_col):
    df["txn_day_of_week"] = df[dt_col].dt.dayofweek.astype("Int64")
    df["is_weekend"] = df["txn_day_of_week"].isin([5, 6]).astype("Int64")
    df["Hour"] = df[dt_col].dt.hour.astype("Int64")
    df["is_night"] = df["Hour"].between(0, 5).astype("Int64")
    return df

In [10]:
finanical_data_process["timestamp"] = pd.to_datetime(finanical_data_process["timestamp"], format="ISO8601")

In [11]:
finanical_data_process["time_since_last_txn_missing"] = finanical_data_process["time_since_last_transaction"].isna().astype("Int64")

In [12]:
finanical_data_process = _add_time_features(finanical_data_process, "timestamp")

In [13]:
finanical_data_process["log_amount"] = np.log1p(finanical_data_process["amount"])

In [14]:
fd_y = finanical_data_process["is_fraud"].astype(bool)

In [15]:
fd_sens = {
    "location": finanical_data_process["location"].copy(),
    "device_used": finanical_data_process["device_used"].copy(),
}

In [16]:
feature_cols = [
    "amount", "transaction_type", "merchant_category", "location", "device_used",
    "time_since_last_transaction", "spending_deviation_score", "velocity_score",
    "geo_anomaly_score", "payment_channel", "time_since_last_txn_missing",
    "txn_day_of_week", "is_weekend", "Hour", "is_night", "log_amount",
]

In [17]:
fd_X = finanical_data_process[feature_cols].copy()

In [18]:
FD_CATEGORICAL = ["transaction_type", "merchant_category", "location", "device_used", "payment_channel"]

In [19]:
enc = OrdinalEncoder()
fd_X[FD_CATEGORICAL] = enc.fit_transform(fd_X[FD_CATEGORICAL])

In [20]:
# Combine Date and Time strings with a separating space
combined_series = banking_data_process['Transaction_Date'] + ' ' + banking_data_process['Transaction_Time']

# Convert to datetime format specifying the explicit layout
banking_data_process['Transaction_DateTime'] = pd.to_datetime(
    combined_series, 
    format='%Y-%m-%d %H:%M', 
    errors='coerce'
)

In [21]:
BD_CATEGORICAL = [
    "Transaction_Type", "Merchant_Category", "Transaction_Location",
    "Customer_Home_Location", "Card_Type", "Is_International_Transaction",
    "Is_New_Merchant", "Unusual_Time_Transaction",
]

In [22]:
for col in BD_CATEGORICAL:
    banking_data_process[col] = banking_data_process[col].fillna("Unknown")

In [23]:
banking_data_process = banking_data_process.dropna(subset=["Fraud_Label", "Transaction_DateTime"])

In [24]:
banking_data_process = _add_time_features(banking_data_process, "Transaction_DateTime")

In [25]:
home_location_raw = banking_data_process["Customer_Home_Location"].copy()

In [26]:
bd_sens = {
    "Customer_Home_Location": banking_data_process["Customer_Home_Location"].copy(),
    "Card_Type": banking_data_process["Card_Type"].copy(),
}

In [27]:
bd_y = banking_data_process["Fraud_Label"].map({"Fraud": True, "Normal": False}).astype(bool)

In [28]:
feature_cols = [
    "Transaction_Amount (in Million)", "Transaction_Type", "Merchant_Category",
    "Transaction_Location", "Customer_Home_Location", "Distance_From_Home",
    "Card_Type", "Account_Balance (in Million)", "Daily_Transaction_Count",
    "Weekly_Transaction_Count", "Avg_Transaction_Amount (in Million)",
    "Max_Transaction_Last_24h (in Million)", "Is_International_Transaction",
    "Is_New_Merchant", "Failed_Transaction_Count", "Unusual_Time_Transaction",
    "Previous_Fraud_Count", "txn_day_of_week", "is_weekend", "Hour", "is_night",
]

In [29]:
bd_X = banking_data_process[feature_cols].copy()

In [30]:
enc = OrdinalEncoder()
bd_X[BD_CATEGORICAL] = enc.fit_transform(bd_X[BD_CATEGORICAL])

In [31]:
CV_FOLDS = 5
TRAIN_SIZES = np.linspace(0.1, 1.0, 7)  # 10%, 25%, 40%, 55%, 70%, 85%, 100%
RANDOM_STATE = 42

In [32]:
# Best hyperparameters found by RandomizedSearchCV in Part 1 (reused, not re-tuned)
FD_DT_PARAMS = {"min_samples_split": 50, "min_samples_leaf": 5, "max_depth": 6,
                 "criterion": "gini", "class_weight": "balanced"}
FD_XGB_PARAMS = {"subsample": 1.0, "n_estimators": 100, "min_child_weight": 3,
                  "max_depth": 4, "learning_rate": 0.03, "colsample_bytree": 0.8}
BD_DT_PARAMS = {"min_samples_split": 2, "min_samples_leaf": 5, "max_depth": 4,
                 "criterion": "entropy", "class_weight": "balanced"}
BD_XGB_PARAMS = {"subsample": 0.8, "n_estimators": 300, "min_child_weight": 3,
                  "max_depth": 4, "learning_rate": 0.01, "colsample_bytree": 0.8}

In [33]:
def fit_predict(X_train, y_train, X_test, dt_params, xgb_params):
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    dt = DecisionTreeClassifier(random_state=RANDOM_STATE, **dt_params)
    dt.fit(X_train, y_train)
    dt_pred = dt.predict(X_test)

    xgb = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight,
                         eval_metric="aucpr", n_jobs=1, **xgb_params)
    xgb.fit(X_train, y_train)
    xgb_pred = xgb.predict(X_test)

    return dt_pred, xgb_pred

In [34]:
def analyse_fairness(y_test, y_pred, sensitive_test, model_name, dataset_name, min_group_size=30):
    metrics = {
        "selection_rate": selection_rate,
        "true_positive_rate": true_positive_rate,
        "false_positive_rate": false_positive_rate,
        "count": count,
    }
    mf = MetricFrame(metrics=metrics, y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_test)

    by_group = mf.by_group.copy()
    by_group["true_fraud_rate"] = pd.Series(y_test.values, index=sensitive_test.index).groupby(sensitive_test).mean()

    # Raw (all groups, however small)
    dpd_raw = demographic_parity_difference(y_test, y_pred, sensitive_features=sensitive_test)
    eod_raw = equalized_odds_difference(y_test, y_pred, sensitive_features=sensitive_test)

    # Filtered: drop groups with too few test-set samples to be statistically meaningful
    small_groups = by_group.index[by_group["count"] < min_group_size].tolist()
    if small_groups:
        mask = ~sensitive_test.isin(small_groups)
        dpd_filt = demographic_parity_difference(y_test[mask], y_pred[mask], sensitive_features=sensitive_test[mask])
        eod_filt = equalized_odds_difference(y_test[mask], y_pred[mask], sensitive_features=sensitive_test[mask])
    else:
        dpd_filt, eod_filt = dpd_raw, eod_raw

    print(f"\n=== {dataset_name} - {model_name} ===")
    print(by_group.round(4))
    print(f"Demographic parity difference (all groups):      {dpd_raw:.4f}")
    print(f"Equalized odds difference (all groups):           {eod_raw:.4f}")
    if small_groups:
        print(f"  -> excluding tiny group(s) {small_groups} (n<{min_group_size}):")
        print(f"Demographic parity difference (filtered):         {dpd_filt:.4f}")
        print(f"Equalized odds difference (filtered):             {eod_filt:.4f}")

    return {
        "by_group": by_group.reset_index().to_dict(orient="records"),
        "small_groups_excluded": small_groups,
        "demographic_parity_difference_all": float(dpd_raw),
        "equalized_odds_difference_all": float(eod_raw),
        "demographic_parity_difference_filtered": float(dpd_filt),
        "equalized_odds_difference_filtered": float(eod_filt),
    }

In [35]:
def plot_fairness(results, dataset_name, groups_col, fname, exclude_groups=None):
    exclude_groups = exclude_groups or []
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, metric, title in zip(
        axes,
        ["true_fraud_rate", "selection_rate", "true_positive_rate", "false_positive_rate"],
        ["True fraud rate", "Selection rate (flagged as fraud)", "True positive rate (recall)", "False positive rate"],
    ):
        dt_df = pd.DataFrame(results["dt"]["by_group"]).set_index(groups_col)
        xgb_df = pd.DataFrame(results["xgb"]["by_group"]).set_index(groups_col)
        dt_df = dt_df.drop(index=exclude_groups, errors="ignore")
        xgb_df = xgb_df.drop(index=exclude_groups, errors="ignore")
        groups = dt_df.index.tolist()
        x = np.arange(len(groups))
        width = 0.35

        if metric == "true_fraud_rate":
            vals = dt_df["true_fraud_rate"]
            ax.bar(x, vals, color="#888888")
        else:
            ax.bar(x - width / 2, dt_df[metric], width, label="Decision Tree", color="#1f77b4")
            ax.bar(x + width / 2, xgb_df[metric], width, label="XGBoost", color="#ff7f0e")
            ax.legend(fontsize=8)

        ax.set_xticks(x)
        ax.set_xticklabels(groups, rotation=45, ha="right", fontsize=8)
        ax.set_title(title, fontsize=10)
        ax.grid(alpha=0.3, axis="y")

    subtitle = f"Fairness analysis by {groups_col} - {dataset_name}"
    if exclude_groups:
        subtitle += f"\n(excludes {exclude_groups} - too few test-set samples to be statistically meaningful)"
    fig.suptitle(subtitle, fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.90 if exclude_groups else 0.94])
    fig.savefig(fname, dpi=150)

In [36]:
def run_dataset_fairness(X, y, sensitive_dict, dt_params, xgb_params, dataset_label, fname_prefix):
    """Fits both models once, then runs the fairness analysis for every
    sensitive attribute supplied in sensitive_dict (e.g. {'location': ..., 'device_used': ...})."""
    # Split X, y, and all sensitive columns together so row alignment is preserved
    sens_names = list(sensitive_dict.keys())
    sens_cols = pd.concat(sensitive_dict.values(), axis=1)
    sens_cols.columns = sens_names

    X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
        X, y, sens_cols, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    dt_pred, xgb_pred = fit_predict(X_train, y_train, X_test, dt_params, xgb_params)

    dataset_results = {}
    for sens_name in sens_names:
        sensitive_test = sens_test[sens_name]
        label = f"{dataset_label} ({sens_name})"

        results = {
            "dt": analyse_fairness(y_test, dt_pred, sensitive_test, "Decision Tree", label),
            "xgb": analyse_fairness(y_test, xgb_pred, sensitive_test, "XGBoost", label),
        }
        dataset_results[sens_name] = results

        exclude = sorted(set(results["dt"]["small_groups_excluded"]) | set(results["xgb"]["small_groups_excluded"]))
        plot_fairness(results, dataset_label, sens_name,
                      f"{fname_prefix}_{sens_name}.png", exclude_groups=exclude)

    return dataset_results

In [37]:
all_results = {}

In [38]:
all_results["financial_dataset"] = run_dataset_fairness(
    fd_X, fd_y, fd_sens, FD_DT_PARAMS, FD_XGB_PARAMS,
    "Financial Transactions Dataset", "fairness_fd_dataset",
)


=== Financial Transactions Dataset (location) - Decision Tree ===
           selection_rate  true_positive_rate  false_positive_rate     count  \
location                                                                       
Berlin             0.8199              0.9974               0.8132  125096.0   
Dubai              0.8163              0.9978               0.8095  124589.0   
London             0.8184              0.9973               0.8119  124990.0   
New York           0.8192              0.9971               0.8127  125227.0   
Singapore          0.8186              0.9966               0.8121  125045.0   
Sydney             0.8190              0.9982               0.8123  124970.0   
Tokyo              0.8199              0.9971               0.8132  125328.0   
Toronto            0.8201              0.9976               0.8134  124755.0   

           true_fraud_rate  
location                    
Berlin              0.0363  
Dubai               0.0362  
London          

In [39]:
all_results["banking_dataset"] = run_dataset_fairness(
    bd_X, bd_y, bd_sens, BD_DT_PARAMS, BD_XGB_PARAMS,
    "Banking Fraud Detection Dataset", "fairness_bd_dataset",
)


=== Banking Fraud Detection Dataset (Customer_Home_Location) - Decision Tree ===
                        selection_rate  true_positive_rate  \
Customer_Home_Location                                       
Faisalabad                      0.4850              0.5500   
Islamabad                       0.4884              0.6355   
Karachi                         0.4836              0.5714   
Lahore                          0.4857              0.6404   
Multan                          0.4887              0.6122   
Unknown                         1.0000              0.0000   

                        false_positive_rate   count  true_fraud_rate  
Customer_Home_Location                                                
Faisalabad                           0.4816  2000.0           0.0500  
Islamabad                            0.4800  1982.0           0.0540  
Karachi                              0.4794  1987.0           0.0458  
Lahore                               0.4786  2030.0           0.04